# Session 4 Lab — Distributions as State, Solved Two Ways

Two parts, in the order of the decks. Part I computes the classical answer keys; Part II trains networks that
carry a distribution, in PyTorch. Each block ends with a **gate**: a bound fixed before the run. Nothing here needs
a network connection or a GPU; from a clean kernel the notebook runs in about a quarter of an hour.

| Block | You will | Deck |
|---|---|---|
| **A** | solve Aiyagari's economy: the endogenous grid method, Young's lottery, a price that clears the market — after making the solver prove a sign change | 4.1 |
| **B** | let the distribution move (Krusell–Smith, graded on a fresh path), and differentiate capital through Young's lottery | 4.2, 4.4 |
| **C** | compute a transition path after a tax change, ask who gains, and solve a life-cycle economy whose interest rate breaks a familiar bracket | 4.3 |
| **D** | Krusell–Smith by DeepHAM: a panel of 50 households, a generalised moment, an actor trained on unrolled paths and a critic fitted by regression | 4.2 |
| **E** | Krusell–Smith by Deep Equilibrium Nets: a cloud of economies, each with its histogram, and a Fischer–Burmeister loss | 4.2 |
| **F** | Krusell–Smith in the sequence space: the last 50 shocks instead of the distribution, and a replay buffer | 4.2 |
| **G** | a life-cycle economy with aggregate risk: a bond market that clears by construction, and a homotopy | 4.3 |
| **H** | a life-cycle economy with income risk and a retirement choice, in the sequence space: continuation values from a history of regimes, a labour market cleared by Newton | 4.3 |

**Calibrations, never mixed.** Blocks A, B2 and C are annual: α = 0.33, β = 0.96, δ = 0.08, CRRA σ = 2, log ε an AR(1)
with persistence 0.90 and innovation s.d. 0.20 on seven Rouwenhorst states, a′ ≥ 0. Blocks B1 and D are Krusell and
Smith's (1998) economy as parameterised by Den Haan, Judd and Juillard (2010). Blocks E, F and G keep the
calibrations of the notebooks they port, and say which; block H keeps the paper's.

Part I descends from the solver of the instructor's quantitative-macro course; Part II trains five networks in
PyTorch — three ports of research teaching notebooks and two written for this lab from the papers' descriptions —
and says where each departs from its source. The numbers the slides quote come from full-budget runs stored in
`session04_assets/recorded_*`; the notebook runs smaller live budgets beside them.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")       # set before numpy is imported; shared machines need it
%matplotlib inline
import json, sys, time
from dataclasses import replace
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

LAB = Path.cwd() if (Path.cwd() / "session04_assets").is_dir() else Path.cwd() / "labs"
ASSETS = LAB / "session04_assets"
sys.path.insert(0, str(ASSETS))
import session04_ha as ha             # block A
import session04_ks as ks             # block B
import session04_paths as paths       # block C

GATE = json.loads((ASSETS / "lab_gates.json").read_text(encoding="utf-8"))
REC = {k: json.loads((ASSETS / f"recorded_{k}.json").read_text(encoding="utf-8")) for k in ("core", "ks", "paths")}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
T_START = time.time()

def gate(name, ok, detail=""):
    print(f"GATE {name}: {'pass' if ok else 'FAIL'}   {detail}")
    assert ok, f"gate {name} failed: {detail}"

print("numpy", np.__version__, "| recorded runs:", ", ".join(REC))

# Part I — The classical benchmarks

The answer keys of Decks 4.1–4.3, computed without a network. Part II trains the networks.

# Block A — Aiyagari: one household, a population, a price


## A1. The economy

A household with assets $a$ and labour productivity $\varepsilon$ chooses $a' \ge \underline a$:
$c + a' = (1+r)a + w\varepsilon$, with Euler **inequality**
$u'(c) \ge \beta(1+r)\,\mathbb E[u'(c')\mid\varepsilon]$, an equality whenever $a' > \underline a$.

In [ ]:
p = ha.Params()
grid, e, Pi, pi = ha.economy(p)
print(p)
print("productivity levels :", np.round(e, 3), "  (mean labour =", round(float(pi @ e), 6), ")")
print("invariant shares    :", np.round(pi, 4))
print(f"asset grid          : {p.n_a} nodes on [{grid[0]:.0f}, {grid[-1]:.0f}], first steps {np.round(np.diff(grid)[:3], 5)} ... last step {np.diff(grid)[-1]:.2f}")

## A2. One sweep by hand, then the whole solve

The grid is put on **tomorrow's** assets. At each $(a',\varepsilon)$ the Euler equation is inverted for today's
consumption; the budget gives the asset level $\tilde a$ that leads there; below $\tilde a(\underline a,\varepsilon)$
the limit binds and consumption is cash on hand. No search, no root-finding.

In [ ]:
r_try = 0.02                                          # a trial price; A8 finds the right one
kl, w = ha.firm(r_try, p)
coh = (1 + r_try) * grid[:, None] + w * e[None, :]
c0 = coh.copy()                                       # first guess: eat everything
c1, ap1 = ha.egm_step(c0, r_try, r_try, w, 0.0, grid, e, Pi, p)
print("after ONE sweep from 'eat everything': households at the lowest productivity save",
      f"{(ap1[:, 0] > 1e-12).mean():.0%} of the time; at the highest, {(ap1[:, -1] > 1e-12).mean():.0%}")

t0 = time.time()
ap, c, sweeps, change = ha.solve_household(r_try, w, 0.0, grid, e, Pi, p)
print(f"converged in {sweeps} sweeps, {time.time() - t0:.2f} s; last change in consumption {change:.1e}")
print(f"work per sweep: {grid.size * e.size:,} inversions.  A grid search would need {grid.size**2 * e.size:,} utility evaluations.")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for q, lab in ((0, "lowest ε"), (p.n_e // 2, "middle ε"), (p.n_e - 1, "highest ε")):
    ax[0].plot(grid, c[:, q], label=lab); ax[1].plot(grid, ap[:, q] - grid, label=lab)
ax[0].set(xlim=(0, 8), ylim=(0, 2.6), xlabel="assets a", title="consumption c(a, ε)"); ax[0].legend(frameon=False)
ax[1].axhline(0, color="0.7", ls="--"); ax[1].set(xlim=(0, 80), xlabel="assets a", title="net saving a' − a")
plt.tight_layout(); plt.show()

## A3. Accuracy where we did not fit

"The policy stopped changing" is a statement about the iteration. The test of the *answer* is the Euler equation
at points **between** the nodes, where the limit is slack: $\log_{10}\lvert 1 - c^{\text{Euler}}/c\rvert$.

In [ ]:
a_off, res, slack = ha.euler_residuals(ap, c, r_try, w, 0.0, grid, e, Pi, p)
worst = np.unravel_index(np.nanargmax(np.where(slack, res, np.nan)), res.shape)
mean_res = float(np.nanmean(np.where(slack, res, np.nan)))
print(f"off-grid Euler residual: mean log10 = {mean_res:.2f}, worst = {res[worst]:.2f} at a = {a_off[worst[0]]:.3f}, productivity state {worst[1] + 1}")
print("the worst point sits just above a kink: linear interpolation cuts the corner there")
plt.figure(figsize=(5.2, 3)); plt.plot(a_off[slack[:, 3]], res[slack[:, 3], 3], lw=0.8)
plt.xlim(0, 60); plt.xlabel("assets a (between nodes)"); plt.ylabel("log10 Euler residual"); plt.title("middle ε"); plt.show()
gate("A3", mean_res < GATE["A_euler_log10_mean_max"] and sweeps < GATE["A_egm_sweeps_max"],
     f"mean log10 residual {mean_res:.2f} (bound {GATE['A_euler_log10_mean_max']}), {sweeps} sweeps")

**Your turn (A1–A3).** (i) Halve and double `n_a` with `replace(p, n_a=...)`: what happens to the residual, and to the
number of sweeps? (ii) Set `sigma=5`: where does the kink move? (iii) Give the household a looser limit,
`a_min=-1.0`, and find the asset level at which it stops binding for the lowest productivity.

## A4. Young's lottery on three nodes

Savings $a'$ almost never land on a node. Split the mass between the two neighbours in proportion to distance:
mass is conserved **and the mean of $a'$ is preserved exactly** — so next period's aggregate capital is exact,
given the policy. Rounding to the nearest node conserves mass and biases every moment.

In [ ]:
nodes = np.array([0.0, 1.0, 3.0])
a_next = np.array([[0.4], [2.5], [2.9]])              # where three households want to go
j, w_hi = ha.lottery(a_next, nodes)
mass = np.array([0.5, 0.3, 0.2])
landed = np.zeros(3)
for i in range(3):
    landed[j[i, 0]] += mass[i] * (1 - w_hi[i, 0]); landed[j[i, 0] + 1] += mass[i] * w_hi[i, 0]
nearest = np.zeros(3); np.add.at(nearest, np.abs(a_next - nodes[None, :]).argmin(axis=1), mass)
print("lottery :", landed.round(4), "| mass", landed.sum().round(12), "| mean", (landed @ nodes).round(12))
print("nearest :", nearest.round(4), "| mass", nearest.sum().round(12), "| mean", (nearest @ nodes).round(12))
print("true mean of a':", float(mass @ a_next[:, 0]))
err = abs(landed @ nodes - mass @ a_next[:, 0])
gate("A4", err < GATE["B_lottery_mean_error_max"], f"the lottery's mean error is {err:.1e}")

## A5. The invariant distribution, two ways

The policy and the productivity chain define one big transition matrix $P_g$ over (assets, productivity) pairs.
The stationary $\lambda$ solves $\lambda = \lambda P_g$: iterate until it stops moving, or solve the linear system.

In [ ]:
t0 = time.time(); lam_it, n_it, _ = ha.stationary_distribution(ap, grid, Pi, pi, method="iterate"); t_it = time.time() - t0
t0 = time.time(); lam_ls, _, _ = ha.stationary_distribution(ap, grid, Pi, pi, method="solve"); t_ls = time.time() - t0
gap = float(np.abs(lam_it - lam_ls).max())
print(f"iteration: {n_it} sweeps, {t_it:.2f} s      linear solve ({lam_ls.size:,} states): {t_ls:.2f} s      largest gap {gap:.1e}")
wealth = lam_it.sum(axis=1)
print(f"at the limit: {wealth[0]:.2%}   mean assets {wealth @ grid:.3f}   mass at the top node {wealth[-1]:.1e}")
plt.figure(figsize=(5.2, 3)); plt.bar(grid[:120], wealth[:120], width=np.diff(grid)[:120], align="edge")
plt.xlabel("assets a"); plt.ylabel("mass on the node"); plt.title(f"invariant distribution at r = {r_try:.0%} (not yet an equilibrium)"); plt.show()
gate("A5", gap < GATE["B_solve_vs_iterate_gap_max"] and wealth[-1] < GATE["B_mass_at_top_max"] and abs(wealth.sum() - 1) < 1e-10,
     f"gap {gap:.1e}; top mass {wealth[-1]:.1e}; total mass − 1 = {wealth.sum() - 1:.1e}")

**Your turn (A4–A5).** (i) Raise the persistence to 0.97 (`replace(p, rho_e=0.97, sd_innov=0.2*np.sqrt((1-0.97**2)/(1-0.9**2)))`):
how many sweeps does iteration need now? (ii) Lower `a_max` to 30 and watch the mass at the top node: at what
point is the grid, not the economics, setting mean assets?

## A6. The capital market, computed

Every point of the supply curve is one household solve and one distribution solve.

In [ ]:
rs = np.array([-0.01, 0.0, 0.01, 0.02, 0.03, 0.035, 0.039])
Ks, Kd, c_ws = [], [], None
for r in rs:
    s_ = ha.capital_supply(float(r), p, grid, e, Pi, pi, c_init=c_ws)
    Ks.append(s_["K_supply"]); Kd.append(s_["K_demand"]); c_ws = s_["c"]
plt.figure(figsize=(5.2, 3.2)); plt.plot(100 * rs, Ks, "o-", label="supply: households"); plt.plot(100 * rs, Kd, "s-", label="demand: the firm")
plt.axvline(100 * (1 / p.beta - 1), color="0.6", ls="--"); plt.ylim(0, 16); plt.xlabel("interest rate r (% a year)"); plt.ylabel("capital K")
plt.legend(frameon=False); plt.title("dashed: 1/β − 1, where supply explodes"); plt.show()

## A7. Make the solver prove a sign change

Bisection halves a bracket. If the root is **not inside** the bracket it still halves it, stops at an edge, and
reports a tidy number. Below, a careless bisection is given a bracket that ends at 2 %. Nothing in its output
looks wrong — until the clearing residual is printed.

In [ ]:
def careless_bisection(lo, hi, steps=40):
    sol = None
    for _ in range(steps):
        mid = 0.5 * (lo + hi)
        sol = ha.capital_supply(mid, p, grid, e, Pi, pi, c_init=None if sol is None else sol["c"])
        if sol["K_supply"] > sol["K_demand"]:
            hi = mid
        else:
            lo = mid
    return sol

bad = careless_bisection(-0.02, 0.02)
bad_resid = abs(bad["K_supply"] - bad["K_demand"]) / bad["K_demand"]
print(f"careless bisection 'converged' to r = {bad['r']:.6%}  <- the upper edge of its bracket")
print(f"   capital supplied {bad['K_supply']:.3f}, demanded {bad['K_demand']:.3f}: the market is off by {bad_resid:.1%}")
gate("A7", bad_resid > GATE["C_naive_residual_min"], "the clearing residual exposes the careless answer")

## A8. The equilibrium, and five checks

`solve_equilibrium` sets the bracket on the **after-tax** return, $(1-\tau_k)\,r < 1/\beta - 1$, evaluates both ends,
and refuses to run without a sign change. Policy and distribution are warm-started from the previous price.

In [ ]:
t0 = time.time()
eq = ha.solve_equilibrium(p)
print(f"excess supply at the ends of the bracket: {eq['sign_change'][0]:+.1f} and {eq['sign_change'][1]:+.1f}  -> a root is inside")
print(f"r* = {eq['r']:.4%}   (1/β − 1 = {1 / p.beta - 1:.4%})   K* = {eq['K_supply']:.4f}   w = {eq['w']:.4f}"
      f"   {eq['bisections']} bisections, {time.time() - t0:.1f} s")
checks = ha.five_checks(eq)
for k_, v_ in checks.items():
    print(f"   {k_:<28} {v_:.3e}" if abs(v_) < 1e-3 else f"   {k_:<28} {v_:.4f}")
rec = REC["core"]["baseline"]
print(f"recorded full run: r* = {rec['r_pct']:.4f}%, K* = {rec['K']:.4f}, wealth Gini {rec['gini_wealth']:.4f}")
print("grid refinement (recorded):")
for row in REC["core"]["refinement"]:
    print(f"   n_a = {row['n_a']:>4}: r* = {row['r_pct']:.4f}%   K* = {row['K']:.4f}   Gini = {row['gini_wealth']:.4f}   mean log10 Euler residual = {row['euler_log10_mean_weighted']:.2f}")
gate("A8", checks["clearing_residual"] < GATE["C_clearing_residual_max"]
     and abs(100 * eq["r"] - rec["r_pct"]) < GATE["C_r_pct_gap_to_recorded_max"] and checks["mass_at_top"] < GATE["B_mass_at_top_max"],
     f"clearing residual {checks['clearing_residual']:.1e}; r* within {abs(100 * eq['r'] - rec['r_pct']):.1e} points of the recorded run")

## A9. What the model gets right, and what it misses

Wealth is far more unequal than earnings from self-insurance alone — and the top of the distribution is missing.
The recorded comparative statics show that this is not a matter of calibration.

In [ ]:
print(f"wealth Gini {checks['gini_wealth']:.3f} against earnings Gini {checks['gini_earnings']:.3f};"
      f"  top 10% hold {checks['top10_share']:.1%}, top 1% hold {checks['top1_share']:.1%}  (US data: Gini 0.82, top 1% about a third)")
print(f"{'economy (recorded)':<32}{'r* %':>8}{'K*':>8}{'Gini':>8}{'at limit %':>12}{'top 1% %':>10}")
for row in REC["core"]["statics"]:
    g = row.get("gini_wealth")
    print(f"{row['label']:<32}{row['r_pct']:>8.2f}{row['K']:>8.2f}" + (f"{g:>8.2f}{100 * row['share_constrained']:>12.1f}{100 * row['top1_share']:>10.1f}" if g else f"{'–':>8}{'–':>12}{'–':>10}"))

**Your turn (A6–A9).** (i) Give `careless_bisection` the bracket `(-0.02, 0.05)`: what happens, and why? (ii) Solve the
economy with `tau_k=0.2` and check that the *pre-tax* equilibrium return exceeds $1/\beta-1$ is impossible only
after tax. (iii) Reproduce one row of the comparative statics live.

# Block B — When the distribution moves: Krusell–Smith

## B1. The Krusell–Smith algorithm

With an aggregate shock the distribution $\Gamma$ becomes a state, and households must forecast it to forecast
prices. Krusell and Smith's answer: forecast only mean capital, $\log K' = \kappa_0(z) + \kappa_1(z)\log K$, and look
for a **fixed point in the forecasting rule**. The loop: household problem on $(a,\varepsilon,K,z)$ given $\kappa$ →
simulate the histogram along one path of $z$ with Young's lottery → regress → update with damping.

The joint transition matrix of $(z,\varepsilon)$ is *built* from the published restrictions and asserted against the
published table. The capital grid must **never clamp** the simulated $K$ — a silent clamp is this block's version
of the careless bracket. The live run starts from the recorded solution and makes a few passes of the loop; the
full run (about a minute) is recorded.

In [ ]:
t0 = time.time()
out = ks.run_short(verbose=True, neural=False)
sol, diag, g = out["sol"], out["diag"], out["gates"]
print(f"\nblock B1 live run: {time.time() - t0:.1f} s")

In [ ]:
rec = REC["ks"]
print("forecasting rule        live:      ", np.round(sol["kappa"], 6).tolist())
print("                        recorded:  ", [np.round(rec["solution"]["kappa"]["bad"], 6).tolist(), np.round(rec["solution"]["kappa"]["good"], 6).tolist()])
pb = rec["published_benchmark"]
print("                        published: ", [pb["kappa_bad"], pb["kappa_good"]], "   (Maliar, Maliar & Valli 2010)")
oos = diag["den_haan_out_of_sample"]; one = diag["one_step_out_of_sample"]
print(f"R² {diag['r2']['bad']:.7f} / {diag['r2']['good']:.7f}   one-step error {one['mean_pct']:.4f}%   Den Haan error on a FRESH path: mean {oos['mean_pct']:.4f}%, max {oos['max_pct']:.4f}%")
bo = sol["settings"].T_oos_burn
K_sim, K_rule = diag["oos"]["K"][bo:], diag["K_rule"]
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3))
ax[0].plot(K_sim[:400], label="simulated economy"); ax[0].plot(K_rule[:400], "--", label="rule iterated, never resynced")
ax[0].set(xlabel="quarter (fresh path)", ylabel="K"); ax[0].legend(frameon=False)
ax[1].plot(diag["err_out"]); ax[1].set(xlabel="quarter (fresh path)", ylabel="% error of the iterated rule")
plt.tight_layout(); plt.show()
gate("B1", all(g["passed"].values()) and oos["max_pct"] < GATE["F_den_haan_oos_max_pct"],       # R² is reported, never gated
     f"all structural gates pass; Den Haan max {oos['max_pct']:.3f}% (bound {GATE['F_den_haan_oos_max_pct']}%)")

**Your turn (B1).** (i) In `ks.SHORT`, narrow the capital grid to (38, 41) with `replace(ks.SHORT, K_lo=38.0, K_hi=41.0)` and
run `ks.solve_ks`: which gate stops you? (ii) Compare `R²` with the Den Haan maximum error across the passes of the loop in
`sol["rows"]`: which one tells you more?

## B2. Differentiating through Young's lottery

Back to block A's annual economy and its equilibrium.

The lottery's weights are piecewise linear in $a'$, so the map *policy parameter → distribution after $T$ periods →
aggregate* can be differentiated exactly by autograd. Take a simple saving rule
$a' = \text{clip}(\theta_0 + \theta_1\cdot\text{cash on hand})$, start from the classical invariant distribution, and ask how
capital and the mass at the limit after $T$ periods respond to $\theta_1$. Three answers to the same question:
autograd through the $T$ lottery steps; a central finite difference of the same deterministic computation; and a
simulated panel of 10,000 households, with and without common random numbers.

One period is a trick question for capital: the lottery preserves the mean, so $K_{t+1}$ is *linear* in the policy and
never exercises the distribution. The mass at the limit, and any $T\ge2$, do.

In [ ]:
import torch
import session04_nn as nnm
torch.set_num_threads(1)
prob = nnm.make_problem(eq)                          # block A's equilibrium: prices and the classical answer key
tab = nnm.lottery_gradients(prob)
print(f"saving rule fitted to the classical policy: θ0 = {tab['theta0']:.3f}, θ1 = {tab['theta1']:.4f}     ({tab['seconds']:.1f} s)")
print(f"\n{'T':>3}{'dK/dθ1 autograd':>18}{'finite diff.':>15}{'panel, common draws':>24}{'panel, independent draws':>28}")
for row in tab["rows"]:
    print(f"{row['T']:>3}{row['d_K_autograd']:>18.4f}{row['d_K_finite_difference']:>15.4f}"
          f"{row['d_K_panel_common_mean']:>15.2f} ± {row['d_K_panel_common_sd']:<6.2f}"
          f"{row['d_K_panel_independent_mean']:>17.1f} ± {row['d_K_panel_independent_sd']:<6.1f}")
print(f"\n{'T':>3}{'d(mass at limit)/dθ1':>22}{'finite diff.':>15}{'panel, common draws':>24}")
for row in tab["rows"]:
    print(f"{row['T']:>3}{row['d_mass_at_limit_autograd']:>22.4f}{row['d_mass_at_limit_finite_difference']:>15.4f}"
          f"{row['d_mass_at_limit_panel_common_mean']:>15.2f} ± {row['d_mass_at_limit_panel_common_sd']:<6.2f}")
print(f"\nclosed form for T = 1: dK/dθ1 = {tab['dK_T1_closed_form']:.4f}  (linear in the policy: the lottery plays no part)")
okE = nnm.check_gate_e(tab)
gate("B2", all(okE.values()), f"autograd and the finite difference agree to {tab['max_relative_error']:.1e} (relative)")

Read the last column first. A simulated panel with **independent** draws puts noise of several times the derivative's
size on it; common random numbers rescue the *capital* derivative but not the mass at the limit, which moves in jumps
of one household. The histogram and the lottery give the same derivative to ten digits, deterministically — which is
what makes it possible to differentiate *through* the evolution of a distribution (Deck 4.4).

**Your turn (B2).** (i) Raise `n_panel` to 100,000: how fast does the independent-draw noise fall? (ii) Differentiate
with respect to $\theta_0$ instead. (iii) Why is the derivative of the mass at the limit exactly zero at $T=1$?

# Block C — A transition path, and who gains

## C1. A capital tax, announced at $t=0$

At $t = 0$ the government announces, without warning and for ever, a 20 % tax on capital income, rebated lump-sum.
Two steady states are not an answer: the economy has to travel from one to the other, and households live through
the journey. The unknown is a **path of capital** $\{K_t\}$: households are solved **backward** from the new steady
state, the distribution is pushed **forward** from the old one, and the path is updated until the capital households
hold equals the capital the prices assumed, period by period.

In [ ]:
t0 = time.time()
tr = paths.transition_path(p, tau_k=0.20, ss_old=eq)          # block A's equilibrium is the starting point
wf = paths.transition_welfare(tr)
print(f"path: {tr['outer_iterations']} outer iterations, {time.time() - t0:.1f} s (including the new steady state)")
print(f"K: {tr['K_old']:.4f} -> {tr['K_new']:.4f} ({100 * (tr['K_new'] / tr['K_old'] - 1):+.2f}%)   "
      f"pre-tax r: {100 * tr['ss_old']['r']:.3f}% -> {100 * tr['ss_new']['r']:.3f}%")
print(f"largest per-period clearing residual {tr['max_residual']:.1e}; goods-market residual {tr['max_resource_residual']:.1e};"
      f" distance of λ_T from the new invariant distribution {tr['terminal_L1']:.1e}")
t_ = np.arange(len(tr["K"]))
fig, ax = plt.subplots(1, 3, figsize=(11, 3))
ax[0].plot(t_, tr["K"]); ax[0].set(title="capital", xlabel="years after the reform")
ax[1].plot(t_, 100 * tr["r"], label="pre-tax"); ax[1].plot(t_, 100 * tr["r_net"], label="after tax"); ax[1].set(title="interest rate, %", xlabel="years"); ax[1].legend(frameon=False)
ax[2].bar(np.arange(1, 11), wf["cev_by_wealth_decile_pct"]); ax[2].axhline(0, color="k", lw=0.6); ax[2].set(title="CEV by wealth decile, %", xlabel="decile of initial wealth")
plt.tight_layout(); plt.show()
print(f"\nmean consumption-equivalent gain ALONG THE PATH (old distribution): {wf['cev_mean_pct']:+.3f}%;  worse off: {wf['share_worse_off_pct']:.1f}% of households")
print(f"comparing the two STEADY STATES only would have reported:            {wf['cev_steady_states_own_distribution_pct']:+.3f}%   <- the sign flips")
sane = paths.cev_sanity(eq, p)
gate("C1", tr["converged"] and tr["max_residual"] < GATE["G_clearing_residual_max"] and tr["max_resource_residual"] < GATE["G_resource_residual_max"]
     and tr["terminal_L1"] < GATE["G_terminal_L1_max"] and max(sane.values()) < GATE["G_cev_sanity_max"],
     "path converged; markets clear every period; λ_T has arrived; CEV passes its sanity checks")

## C2. A life-cycle economy, and a bracket that fails

Sixty cohorts, forty-five working years, no bequests, a pay-as-you-go pension financed by a payroll tax that balances
the budget each period. Households are solved backward over **age** — there is no inner fixed point — and the
cohorts *are* the distribution. With finite lives $1/\beta - 1$ is **not** an upper bound on the interest rate.
A bisection that assumes it stops at the edge of its bracket and reports a number; only the sign-change test and
the clearing residual show that nothing was solved.

In [ ]:
setup = paths.olg_setup(p)
lo, hi = 0.005, 1 / p.beta - 1 - 1e-6                  # the bracket that is right for infinitely-lived households
for _ in range(40):
    mid = 0.5 * (lo + hi); o_ = paths.olg_at_r(mid, 0.40, setup)
    lo, hi = (lo, mid) if o_["K_supply"] > o_["K_demand"] else (mid, hi)
careless = paths.olg_at_r(0.5 * (lo + hi), 0.40, setup)
res_c = abs(careless["K_supply"] - careless["K_demand"]) / careless["K_demand"]
print(f"careless bracket: r = {100 * careless['r']:.4f}%  (= 1/β − 1 to six digits), market off by {res_c:.1%}")

rows = paths.olg_comparison(p, rates=(0.40, 0.0), setup=setup)
for row in rows:
    o = row["solution"]
    print(f"replacement {row['replacement']:.2f}: payroll tax {row['tau_p']:.4f}   r* = {row['r_pct']:.3f}%   K = {row['K']:.3f}   "
          f"clearing residual {o['clearing_residual']:.1e}   pension budget {o['budget_residual']:.1e}")
o = rows[0]["solution"]
print(f"with the pension r* − (1/β − 1) = {rows[0]['r_pct'] - 100 * (1 / p.beta - 1):+.3f} points: above the 'bound'")
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(o["profile_age"], o["profile_assets"], label="with pension"); ax[0].plot(o["profile_age"], rows[1]["solution"]["profile_assets"], "--", label="no pension")
ax[0].set(xlabel="age", title="mean assets by age"); ax[0].legend(frameon=False)
ax[1].plot(o["profile_age"], o["profile_consumption"], label="consumption"); ax[1].plot(o["profile_age"], o["profile_net_income"], label="net income"); ax[1].set(xlabel="age"); ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()
gate("C2", o["clearing_residual"] < GATE["G_olg_clearing_residual_max"] and abs(o["budget_residual"]) < GATE["G_olg_budget_residual_max"] and res_c > GATE["C_naive_residual_min"],
     "the life-cycle economy clears, the pension budget balances, and the careless bracket is exposed")

**Your turn (C).** (i) Re-run the path with `T=60`: which diagnostic tells you the horizon is too short? (ii) Make
the tax 10 %: does the sign of the steady-state comparison still differ from the sign along the path? (iii) Set the
replacement rate to 0.2 and place $r^*$ relative to $1/\beta-1$.

# Part II — Networks that carry a distribution, in PyTorch

Five networks, in the order of the decks: three ports of research teaching notebooks (E, F, G) and two written
for this lab from the papers' descriptions (D, H). Each is trained live for a few CPU-minutes and set beside a
longer run recorded with the same code. `USE_RECORDED = True` skips the live training and grades the recorded
network instead. The gates were fixed from five-seed pilots of the live settings, before this notebook was run.

In [ ]:
import torch
torch.set_num_threads(4)
import session04_deepham_ks as dh       # block D
import session04_deqn_ks as dq          # block E
import session04_seqspace_ks as sq      # block F
import session04_olg_homotopy as og     # block G
import session04_seqspace_olg as so     # block H
USE_RECORDED = False
print("torch", torch.__version__)

# Block D — Krusell–Smith by DeepHAM

Deck 4.2, Method 1. A **panel**: $E$ economies side by side, each a sample of $N=50$ households with their own
wealth and employment draws; prices are marginal products at the panel's mean wealth. Three networks: a
**generalised moment** $Q(a)$, averaged over the panel into the one number $\bar Q$ that the others read; an
**actor** that maps $(a_i, e_i, z, \bar Q)$ to a consumption share of cash on hand, so saving is never negative; a
**critic** that maps the same inputs to lifetime utility. The actor is trained on simulated paths: from a stored
cross-section, unroll $T_{\mathrm{unroll}}$ periods with fresh shocks, add $\beta^{T}$ times the critic at the end, and
climb household 0's discounted utility; the other 49 use the same network with the gradient stopped (fictitious
play). The critic is fitted by regression on realised discounted utility. Three clocks: an actor update every
mini-batch of $B$ pool rows; a new pool every $U_{\mathrm{pool}}=P/B$ updates, when the old one is exhausted; a
critic refit every $F$ updates, after which every run replaces its economy's stored state with its last period.

**How this differs from the paper.** Written for this lab from Deck 4.2's description of the algorithm and the
settings of the public repository, not a port of its code. The economy is block B's (Den Haan, Judd and
Juillard's Krusell–Smith). One critic net instead of three averaged; the moment net is trained with the actor
only; prices use the model's unemployment rate $u(z)$, not the panel's realised employment; and the actor is
**warm-started** on block B's classical policy (the paper starts from a constant consumption share), as block E
pre-fits to EGM — from a random start the panel runs away before the critic can value anything.

**Scaled for a CPU.** The paper: 384 economies, a burn-in of 6,000 periods, pool runs of 450, an unroll of 150,
10,000 updates, critic runs of 2,000 with 800-period targets, 201 passes, refits every 2,000 updates. Here:
$E=32$, burn-in 300, pool runs of 120 ($P=480$ rows, $B=48$), unroll 60 ($\beta^{60}\approx0.55$ is left to the
critic), $U=300$, critic runs of 450 with 150-period targets ($\beta^{150}\approx0.22$ of a target is cut off), 50
passes, refits every 100. Every field of `dh.Config` carries the paper's value beside it.

In [ ]:
polD = dh.ClassicalPolicy()                  # block B's household policy under its recorded rule
print(f"classical rule: K = {polD.K_ss:.3f}; its household policy solved in {polD.sweeps} EGM sweeps")
t0 = time.time()
if USE_RECORDED:
    modelD, histD, cfgD = dh.load(ASSETS / "recorded_deepham.pt")
else:
    cfgD = dh.LIVE
    modelD, histD = dh.train(cfgD, verbose=True, pol=polD)
print(f"block D training: {time.time() - t0:.0f} s")

In [ ]:
sD = dh.summary(modelD, histD, cfgD, polD)
recD = torch.load(ASSETS / "recorded_deepham.pt", weights_only=False)["history"]["summary"]
print(f"{'':<12}{'Euler, mean |err|':>18}{'p90':>8}{'K vs classical':>16}{'critic R2':>11}{'CEV vs classical':>18}")
for name, s_ in (("this run", sD), ("recorded", recD)):
    print(f"{name:<12}{s_['euler_mean_abs']:>18.4f}{s_['euler_p90_abs']:>8.4f}{100 * (s_['K_mean'] / s_['K_ss_classical'] - 1):>15.2f}%"
          f"{s_['critic_r2_last']:>11.3f}{100 * s_['cev_vs_classical']:>17.2f}%")

pdD = dh.share_plot_data(modelD, cfgD, polD); eeD = dh.euler_errors(modelD, cfgD, polD)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.1))
for ee, lab in ((0, "unemployed"), (1, "employed")):
    ax[0].plot(pdD["a"], pdD["share"][ee][0], label=f"actor, {lab}"); ax[0].plot(pdD["a"], pdD["share"][ee][1], "k:", lw=0.8)
ax[0].set(xlabel="wealth a", ylabel="consumption share of cash", title="actor vs classical (dotted)"); ax[0].legend(frameon=False, fontsize=7)
ax[1].plot(eeD["K_path"].T, lw=0.6); ax[1].axhline(polD.K_ss, color="k", lw=0.8); ax[1].set(xlabel="period", title="K in 16 fresh economies")
ax[2].plot(histD["objective"], lw=0.8); ax[2].set(xlabel="actor update", title="objective: unroll + critic")
for u in histD["refit_at"][1:]:
    ax[2].axvline(u, color="r", lw=0.6, ls=":")
plt.tight_layout(); plt.show()
gate("D", sD["euler_mean_abs"] < GATE["DH_euler_mean_abs_max"] and sD["euler_p90_abs"] < GATE["DH_euler_p90_abs_max"]
     and abs(sD["K_mean"] / sD["K_ss_classical"] - 1) < GATE["DH_K_rel_dev_max"] and sD["critic_r2_last"] > GATE["DH_critic_r2_min"]
     and sD["cev_vs_classical"] > GATE["DH_cev_vs_classical_min"] and sD["objective"]["change"] > GATE["DH_objective_change_min"],
     f"Euler mean |err| {sD['euler_mean_abs']:.4f}; K {100 * (sD['K_mean'] / sD['K_ss_classical'] - 1):+.2f}% from the classical K; "
     f"critic R2 {sD['critic_r2_last']:.3f}; CEV vs classical {100 * sD['cev_vs_classical']:+.2f}%")

**Your turn (D).** (i) Shorten the unroll, `replace(dh.LIVE, T_unroll=20)`: more of the objective rests on the
critic — what happens to the Euler error? (ii) Replace the learned moment by the mean, `modelD.qbar = lambda a:
a.mean(1, keepdim=True) / 50` — Krusell and Smith's bet; retrain the actor: does the welfare gap move? (iii) Stop
the refits, `replace(dh.LIVE, F=10**6)`: the actor climbs a critic fitted under the warm-started policy only;
which number warns you first?

# Block E — Krusell–Smith by Deep Equilibrium Nets

Deck 4.2, Method 2. A **cloud** of economies, each carrying its whole histogram over $(a,e)$, moves one period per
gradient step; the loss is the Euler condition of households drawn from those economies, written with the
Fischer–Burmeister function so the borrowing limit is one equation. The network reads $(a, e, Z, K)$ and returns
a saving *rate*, so feasibility holds by construction; tomorrow's capital comes from pushing each histogram
through Young's lottery under the current policy.

**How this differs from Deck 4.2's Method 2.** The deck's version (the public continuum-of-agents code) reads two
full histograms and also outputs a multiplier. This notebook's network reads only mean capital $K$ — the
Krusell–Smith bet — and outputs one saving rate. Prices are marginal products at $(K, Z)$; there is no price network.

**What was changed in the source notebook, and why.** Run as written, it does not produce a valid steady state:
(1) TFP levels were exponentiated twice, so $Z\approx2.72$; (2) the asset grid stopped at 50, where 15 % of the
mass piled up on the top node; (3) its Fischer–Burmeister function was defined but never used, so the loss
forced the Euler equation to hold with equality at the limit; (4) saving above the top node was not clamped. All
four are fixed here. Two training changes were also needed — a warm start fitted to the classical policy, and
tomorrow's consumption read from a frozen copy of the network (Session 3's target network) — without which
the cloud runs away from the steady state although the loss is right. The full list is in the module's docstring.

In [ ]:
ss = dq.steady_state(verbose=False)
print(f"classical steady state (EGM + Young, sign-change bisection): K = {ss['K']:.4f}, r = {100 * ss['r']:.3f}%")
t0 = time.time()
if USE_RECORDED:
    recE = torch.load(ASSETS / "recorded_deqn.pt", weights_only=False)
    netE = dq.KS_Network(n_hidden=recE["cfg"]["n_hidden"]); netE.load_state_dict(recE["state_dict"])
    modelE = dq.KrusellSmithModel(ss, ss.get("cal", dq.CAL)); histE = recE["history"]
else:
    netE, modelE, histE = dq.train(seed=0, cfg=dq.SHORT, ss=ss, verbose=True)
print(f"block E training: {time.time() - t0:.0f} s")

In [ ]:
eeE = dq.euler_errors(netE, modelE)
pgE = dq.policy_gap_vs_egm(netE, modelE, ss)
KE, zE, _ = dq.simulate_K_path(netE, modelE, T=500, seed=7, n_econ=16)
devE = KE[:, 100:].mean() / modelE.K_ss - 1
recE = torch.load(ASSETS / "recorded_deqn.pt", weights_only=False)["summary"]
print(f"{'':<12}{'log10 Euler, weighted':>23}{'p90':>8}{'K vs K_ss':>12}{'policy gap':>12}")
print(f"{'this run':<12}{eeE['euler_log10_weighted_mean']:>23.2f}{eeE['euler_log10_p90']:>8.2f}{100 * devE:>11.2f}%{pgE['weighted_mean']:>12.3f}")
print(f"{'recorded':<12}{recE['euler']['euler_log10_weighted_mean']:>23.2f}{recE['euler']['euler_log10_p90']:>8.2f}"
      f"{100 * recE['K_sim_rel_dev_mean']:>11.2f}%{recE['policy_gap_vs_egm']['weighted_mean']:>12.3f}")

pdE = dq.policy_plot_data(netE, modelE, ss)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.1))
for j in range(len(pdE["e"])):
    ax[0].plot(pdE["a"], pdE["sav_net"][:, j] - pdE["a"], label=f"network, e={pdE['e'][j]:.2f}")
    ax[0].plot(pdE["a"], pdE["sav_egm"][:, j] - pdE["a"], "k:", lw=0.8)
ax[0].set(xlim=(0, 80), xlabel="assets a", title="net saving at K_ss (dotted: EGM)"); ax[0].legend(frameon=False, fontsize=7)
ax[1].plot(KE.T, lw=0.6); ax[1].axhline(modelE.K_ss, color="k", lw=0.8); ax[1].set(xlabel="period", title="K in 16 simulated economies")
ax[2].semilogy(histE["loss"]); ax[2].set(xlabel="period of the cloud", title="FB² loss")
plt.tight_layout(); plt.show()
gate("E", eeE["euler_log10_weighted_mean"] < GATE["D_euler_log10_weighted_max"] and eeE["euler_log10_p90"] < GATE["D_euler_log10_p90_max"]
     and abs(devE) < GATE["D_K_rel_dev_max"] and pgE["weighted_mean"] < GATE["D_policy_gap_max"],
     f"Euler {eeE['euler_log10_weighted_mean']:.2f} (p90 {eeE['euler_log10_p90']:.2f}); simulated K {100 * devE:+.2f}% from K_ss; policy gap {pgE['weighted_mean']:.3f}")

**Your turn (E).** (i) Train from the source's random start, `dq.train(0, cfg=dq.RANDOM, ss=ss)`: where does the cloud
go, and what happens to the residual? (ii) Set `target_sync=0` with `replace(dq.SHORT, target_sync=0)`: which number
warns you first? (iii) The network sees only $K$. Which experiment would tell you whether that is enough?

# Block F — Krusell–Smith in the sequence space

Deck 4.2, Method 3. Forget the distribution: the network reads the **last 50 aggregate shocks** and returns the
whole consumption function on a 101-point grid, built from an I-spline basis so that the marginal propensity to
consume is positive and falling by construction. Histograms are still pushed forward by Young's lottery, but only
to produce prices and training states, which are stored in a replay buffer of 128. The loss is the
Fischer–Burmeister residual at every grid node of every stored state. This is a faithful port of the authors'
teaching notebook (its calibration: $\delta=0.10$, two income and two aggregate states).

In [ ]:
t0 = time.time()
if USE_RECORDED:
    netF, histF, bufF, _ = sq.load(ASSETS / "recorded_seqspace.pt")
else:
    netF, histF, bufF = sq.train(seed=0, cfg=sq.LIVE, verbose=True)
print(f"block F training: {time.time() - t0:.0f} s")

In [ ]:
sF = sq.summarise(sq.euler_diagnostics(netF, bufF))
_, _, bufR, _ = sq.load(ASSETS / "recorded_seqspace.pt")
netR = sq.load(ASSETS / "recorded_seqspace.pt")[0]
sR = sq.summarise(sq.euler_diagnostics(netR, bufR))
print(f"{'':<26}{'FB² (buffer)':>14}{'|FB| p50':>11}{'p90':>10}{'mean K':>9}{'clearing gap':>14}")
for name, s_ in (("this run", sF), ("recorded, 10,000 epochs", sR)):
    print(f"{name:<26}{s_['fb2_buffer']:>14.2e}{s_['p50_abs_fb']:>11.1e}{s_['p90_abs_fb']:>10.1e}{s_['K_mean']:>9.2f}{s_['max_mc_gap']:>14.1e}")
print("authors' notebook (JAX), 10,000 epochs: FB² 3.1e-7, mean K 4.82")

pol = sq.policy_on_reference_histories(netF, bufF); simF = sq.simulate(netF, T=500, seed=3)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.1))
for zi, zl in enumerate(("all-low history", "all-high history")):
    for ei, el in enumerate(("low ε", "high ε")):
        ax[0].plot(pol["k"], pol["mpc"][zi, ei], label=f"{zl}, {el}")
ax[0].set(xlim=(0, 20), xlabel="k", title="marginal propensity to consume"); ax[0].legend(frameon=False, fontsize=6)
ax[1].plot(simF["K"]); ax[1].set(xlabel="period", title="K, one simulated economy")
h_ = np.array(histF); ax[2].semilogy(h_[:, 0], h_[:, 1]); ax[2].set(xlabel="epoch", title="FB² (last minibatch)")
plt.tight_layout(); plt.show()
lo, hi = GATE["E_K_mean_range"]
gate("F", sF["fb2_buffer"] < GATE["E_fb2_buffer_max"] and sF["p50_abs_fb"] < GATE["E_p50_abs_fb_max"] and sF["p90_abs_fb"] < GATE["E_p90_abs_fb_max"]
     and sF["max_mc_gap"] < GATE["E_max_mc_gap_max"] and lo < sF["K_mean"] < hi,
     f"FB² {sF['fb2_buffer']:.1e} over the buffer; |FB| p90 {sF['p90_abs_fb']:.1e}; mean K {sF['K_mean']:.2f}")

**Your turn (F).** (i) Shorten the history to `H = 10` (edit `sq.H` before building a new `sq.Actor`): what happens to
the residual after a long run of good shocks? (ii) Why can the clearing gap be at round-off here although no
equation imposes it? (iii) Compare the MPC at $k=0$ under the two histories: which household is constrained?

# Block G — Cohorts, aggregate risk, and a market that clears by construction

Deck 4.3, Part B. Seven cohorts, capital and a one-period bond, TFP risk integrated by Gauss–Hermite quadrature.
Without idiosyncratic risk each cohort is one household, so the state is 15 numbers. One network returns six
capital choices, six bond choices and the bond price. Two ideas from the authors' paper:

* a **market-clearing layer**: subtract the mean excess demand from every bond choice, so $\sum_j b'_j=0$ holds
  for *any* weights — the bond market is not learned, it is imposed;
* a **homotopy**: solve the capital-only economy, pretrain the bond price on it, then let bonds in and relax the
  borrowing limit step by step to its target, training at every step.

**Scaled for a CPU.** Deck 4.3 quotes the authors' settings (8,192 states, learning rate $10^{-5}$, 4,096 homotopy
steps). The live run here uses 1,024 states, a rate of $10^{-3}$, and $32\times3$ relaxation episodes; the recorded
run doubles the cloud and triples the episodes. The economy is the same.

In [ ]:
t0 = time.time()
solG = og.load(ASSETS / "recorded_olg.pt") if USE_RECORDED else og.Solver(cfg=og.LIVE).run(verbose=True)
print(f"block G training: {time.time() - t0:.0f} s")

In [ ]:
dG = og.diagnostics(solG)                 # a fresh simulation at the true shock size
drops = og.stage_drops(solG.log)
dR = og.diagnostics(og.load(ASSETS / "recorded_olg.pt"))
print(f"{'':<12}{'capital KKT, mean %':>21}{'p99 %':>8}{'bond KKT, mean %':>18}{'p99 %':>8}{'bond excess':>13}{'K':>7}{'bond price':>12}")
for name, d_ in (("this run", dG), ("recorded", dR)):
    print(f"{name:<12}{d_['kee_mean_all']:>21.3f}{d_['kee_p99_all']:>8.3f}{d_['bee_mean_all']:>18.3f}{d_['bee_p99_all']:>8.3f}"
          f"{d_['bond_excess_max']:>13.1e}{d_['K_mean']:>7.3f}{d_['p_b_mean']:>12.4f}")
print("\nloss, last tenth / first tenth of each stage:", {k: round(v["ratio"], 3) for k, v in drops.items()})
print("relax rises in every seed: each step moves the constraint. It is checked for blow-up, not for a fall.")
og.plot(dG, solG.log); plt.show()
(klo, khi), (plo, phi) = GATE["F_K_range"], GATE["F_p_b_range"]
gate("G", dG["bond_excess_max"] < GATE["F_bond_excess_max"]
     and dG["kee_mean_all"] < GATE["F_kee_mean_max"] and dG["kee_p99_all"] < GATE["F_kee_p99_max"]
     and dG["bee_mean_all"] < GATE["F_bee_mean_max"] and dG["bee_p99_all"] < GATE["F_bee_p99_max"]
     and drops["capital"]["ratio"] < GATE["F_ratio_capital_max"] and drops["first"]["ratio"] < GATE["F_ratio_first_max"]
     and drops["fine"]["ratio"] < GATE["F_ratio_fine_max"] and drops["relax"]["ratio"] < GATE["F_ratio_relax_max"]
     and klo < dG["K_mean"] < khi and plo < dG["p_b_mean"] < phi,
     f"bond market clears to {dG['bond_excess_max']:.0e}; KKT errors {dG['kee_mean_all']:.3f}% / {dG['bee_mean_all']:.3f}% (means)")

**Your turn (G).** (i) Skip the homotopy: build `og.Config` with `steps_pre=0, homotopy_steps=1` and start bonds at
the target limit. What does the loss do? (ii) Remove the market-clearing layer's adjustment (set the bond mask to 1
and drop the subtraction in `Model.S`): which residual has to carry the market now? (iii) Who borrows, at which
ages, and why does the bond price sit near one half?

# Block H — The retirement OLG in the sequence space

Deck 4.3, Part C. Ten cohorts of six years; ages 1–6 work, age 7 chooses to work or retire (a utility cost of
working, a logit taste shock), 8–10 are retired on home income; three income states, dynastic; one asset; three
aggregate regimes, the third a disaster. Every cohort is a histogram $\Gamma_j(e,k)$, so the state is $J N_e N_k$
masses — and the network never reads it. It reads the **last $T_{\mathrm{hist}}$ regimes** and returns the
**continuation values** $\hat W(j,e,k')$ on the asset grid (a first value plus softplus increments, so $\hat W$ rises
in $k'$) and a guess $\hat\lambda$ of the share of age 7's labour supplied. Inside a period: clear the labour market
by Newton on $ED(L)$ with the derivative by autograd (only the retirement choice depends on $L$), solve every
cohort, move the histograms by Young's lottery age by age — the age-7 split by $p^{\mathrm{work}}$, the dying
cohort's bequests to the newborns. Targets: $W^{\mathrm{tar}}(j,e,k')=\sum_{z'}\Pi^z\sum_{e'}\Pi^e\,V_{t+1}(j{+}1,e',k'\mid z')$,
with $t+1$ solved under a **target network**, an exponential moving average of the online one; the loss is the
mean squared relative error on $W$'s first values, its increments and $\lambda$. $E=B$ pairs run side by side;
every update uses all of them and then moves each one period: no pool, no epochs, nothing reused. Three stages:
a deterministic steady state (a bisection on $K$ with a sign change asserted) seeds the pairs; pretraining with
the continuation held at the steady state's; the full loop.

**How this differs from the paper.** Written for this lab from §6 of the paper, which publishes no code for this
economy. Inside the period the saving problems are solved by **direct maximisation on a fine saving grid**
($N_f=200$ points, the continuation value interpolated between asset-grid points) instead of generalised EGM
with an upper-envelope step: the maximum over the grid is global, so the non-concave continuation values around
the retirement choice need no envelope; the price is a policy that is exact only on the fine grid. The network
is first regressed on the steady state's $W$ (the lab's warm start, as block E's). Starting histories are all-normal.

**Scaled for a CPU.** The paper: 300 asset points, a window of 100, 1,024 pairs, three layers of 1,024, Adam at
$10^{-6}$, $\tau=10^{-4}$. Here: $N_k=40$ (dense near zero), $T_{\mathrm{hist}}=20$ (a lifetime is 10), $E=B=64$,
two layers of 128, Adam at $3\times10^{-4}$, $\tau=10^{-2}$, 60 + 300 updates with $\varepsilon_{\mathrm{loss}}$ 1, then
0.1, then 0.01. Every field of `so.Config` carries the paper's value beside it.

In [ ]:
t0 = time.time()
trH = so.load(ASSETS / "recorded_seqspace_olg.pt") if USE_RECORDED else so.Trainer(so.LIVE).run(verbose=True)
print(f"block H training: {time.time() - t0:.0f} s")

In [ ]:
sH = so.summary(trH)
recH = json.loads((ASSETS / "recorded_seqspace_olg.json").read_text(encoding="utf-8"))["summary"]
print(f"{'':<12}{'rel. err, first':>16}{'increments':>12}{'p90':>8}{'|lam^ - lam|':>13}{'max |ED|':>10}{'K mean':>9}{'loss ratio':>12}")
for name, s_ in (("this run", sH), ("recorded", recH)):
    print(f"{name:<12}{s_['rel_first_mean']:>16.3f}{s_['rel_inc_mean']:>12.3f}{s_['rel_inc_p90']:>8.3f}{s_['lam_abs_err']:>13.3f}"
          f"{s_['ED_max']:>10.1e}{s_['K_mean']:>9.4f}{s_['loss_drop']['ratio']:>12.3f}")
print(f"steady state: K {sH['ss_K']:.4f}, L {sH['ss_L']:.4f}, share of age 7's labour supplied {sH['ss_lambda']:.3f}; "
      f"clearing residual {sH['ss_residual']:.0e}")

picH = so.policy_pictures(trH); simH = so.simulate(trH, T=200, seed=5)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.1))
for name, ls in (("normal", "-"), ("disaster", "--")):
    for ie, e_ in enumerate((0.5, 1.0, 1.5)):
        ax[0].plot(picH["k"], picH[name]["p_work"][ie], ls, lw=0.9, label=f"{name}, e = {e_}")
ax[0].set(xlabel="assets at age 7", ylabel="probability of working", title="the retirement choice"); ax[0].legend(frameon=False, fontsize=6)
for name, ls in (("normal", "-"), ("disaster", "--")):
    ax[1].plot(picH["k"], picH[name]["c_before"][1], ls, lw=0.9, label=name)
ax[1].set(xlabel="assets at age 6", ylabel="consumption, e = 1", title="the age before the choice"); ax[1].legend(frameon=False, fontsize=7)
ax[2].plot(simH["K"][0], lw=0.8); dis = np.nonzero(simH["z"][0] == 2)[0]; ax[2].plot(dis, simH["K"][0][dis], "r.", ms=4)
ax[2].axhline(sH["ss_K"], color="k", lw=0.8); ax[2].set(xlabel="period", title="K on one path (red: disasters)")
plt.tight_layout(); plt.show()
klo, khi = GATE["SO_K_range"]
gate("H", sH["ss_residual"] < GATE["SO_ss_residual_max"] and sH["ss_mass_error"] < GATE["SO_ss_mass_error_max"]
     and max(sH["ED_max"], sH["ED_max_sim"]) < GATE["SO_ED_max"] and sH["rel_first_mean"] < GATE["SO_rel_first_mean_max"]
     and sH["rel_inc_mean"] < GATE["SO_rel_inc_mean_max"] and sH["rel_inc_p90"] < GATE["SO_rel_inc_p90_max"]
     and sH["lam_abs_err"] < GATE["SO_lam_abs_err_max"] and klo < sH["K_mean"] < khi and sH["loss_drop"]["ratio"] < GATE["SO_loss_ratio_max"],
     f"steady state clears to {sH['ss_residual']:.0e}; labour clears to {max(sH['ED_max'], sH['ED_max_sim']):.0e}; "
     f"relative errors {sH['rel_first_mean']:.3f} / {sH['rel_inc_mean']:.3f}; K {sH['K_mean']:.4f}")

**Your turn (H).** (i) A window of one lifetime, `replace(so.LIVE, T_hist=10)`: what changes in the errors in the
periods after a disaster? (ii) No target network, `replace(so.LIVE, tau=1.0)` (the online network is its own
target): does the loss still fall? (iii) Set `so.WORK_COST = 0.0` before training: everyone works at age 7 and the
continuation values are concave again — is the fine saving grid still needed?

---
**What the gates were.** A: the Euler equation off the grid, mass at the top of the grid, a sign change before a
bisection, the clearing residual. B: a grid that never clamps, a forecast graded on a path it never saw, a derivative
checked two ways. C: markets clearing in every period, a horizon long enough, welfare that is zero when nothing
changes. D: a Euler residual on a fresh panel, the capital a panel implies against the classical rule's, a critic that
fits rows it was not trained on, no welfare lost against the classical policy. E: the capital a network implies, and
its policy against the classical one. F: the residual over every stored state, not the last minibatch. G: a market
cleared to round-off by construction, errors on a fresh simulation, a loss that falls where it should. H: a steady
state that clears after a proven sign change, a labour market cleared to round-off at every state, the network
against its own targets, a loss that falls. None of them is a loss value alone.

In [ ]:
print(f"notebook run time: {time.time() - T_START:.0f} s")